This project is based on Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD by Local Inference Lab, Inc., a non-profit organization, available at [https://huggingface.co/local-inference-lab/Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD](https://huggingface.co/local-inference-lab/Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD). Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD is licensed under the Local Inference Lab License, Version 1.0.

In [ ]:
# FLARE stage-3e smoke (card: SMOKE3E-CARD.md). 3d ran clean at r=0.45: Flare snapshots its prefix cache at request
# boundaries, so the moving 4-image window (each turn strips the previous image) breaks every new turn's prefix.
# 3e replaces EVERY image with a fixed placeholder, so prompts stay prefix-stable: Flare's ceiling with its cache
# working. S=10 and 14 only. Stage-3d header follows.
# FLARE stage-3d smoke (card: SMOKE3D-CARD.md). 3c served (attr 11 skipped) but every request got 400: Flare allows
# max_images_per_prompt 4 and the logged requests carry a median of 24. 3d sends the replay with history images
# stripped exactly as ARC3_HISTORY_IMAGE_KEEP=0 would (current turn keeps its <=3 images). Stage-3c header follows.
# FLARE stage-3c smoke (card: SMOKE3C-CARD.md). 3b loaded the weights (staged parts + flare_ra.so) and then failed in
# "Prepare execution": cuGraphKernelNodeSetAttribute -> CUDA_ERROR_INVALID_VALUE on driver 580. 3c also wraps that call
# (flare_ra.so dlsym override): every attribute id is logged, and ones the driver rejects are skipped.
# Stage-3b header follows.
# FLARE stage-3b smoke on the Kaggle RTX PRO 6000 (card: ../SMOKE-CARD.md, ../SMOKE3-CARD.md, SMOKE3B-CARD.md).
# Stage 3 never got past "Load weights": Flare reads its package with small O_DIRECT reads, which on
# the gcsfuse dataset mount ran at 1.7 MB/s (4.0 GB in 40 min). 3b preloads flare_ra.so
# (../readahead/), which drops O_DIRECT on the package and serves Flare's reads from a block cache fed
# by 16 threads of 64 MiB fresh-handle reads, the pattern that precaches datasets at ~800 MB/s on
# these machines. A 20 s raw read benchmark of that pattern runs first. Everything else is stage 3.
import glob, hashlib, json, os, re, shutil, signal, subprocess, sys, threading, time, urllib.request
from pathlib import Path

T0 = time.monotonic()
DEADLINE = T0 + 110 * 60          # self-imposed session cap (card)
OUT = Path('/kaggle/working/flare'); OUT.mkdir(parents=True, exist_ok=True)
RESULT = {'started_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
TOTAL = 107011338717
PUBLISHER = 'local-inference-lab'
PORT = 8002
MODEL = 'flashnext'

def log(msg):
    line = f'[{time.monotonic() - T0:7.1f}s] {msg}'
    print(line, flush=True)
    with open(OUT / 'smoke.log', 'a') as f:
        f.write(line + '\n')

def save():
    (OUT / 'result.json').write_text(json.dumps(RESULT, indent=1, default=str))

def mem_gib():
    try:
        return int(open('/sys/fs/cgroup/memory.current').read()) / 2**30
    except Exception:
        return -1.0

WATCH = {'peak_gib': 0.0, 'killed': False, 'proc': None}
def watchdog():
    last = 0.0
    while True:
        m = mem_gib(); WATCH['peak_gib'] = max(WATCH['peak_gib'], m)
        if m > 160 and WATCH['proc'] is not None and WATCH['proc'].poll() is None and not WATCH['killed']:
            WATCH['killed'] = True
            log(f'WATCHDOG: cgroup memory {m:.1f} GiB > 160; killing flare')
            try:
                os.killpg(WATCH['proc'].pid, signal.SIGKILL)
            except Exception as e:
                log(f'watchdog kill failed: {e!r}')
        if time.monotonic() - last > 30:
            last = time.monotonic()
            tail = ''
            try:
                tail = (OUT / 'flare-server.log').read_bytes()[-300:].decode(errors='replace').replace(chr(10), ' | ')
            except Exception:
                pass
            io = ''
            try:
                if WATCH['proc'] is not None and WATCH['proc'].poll() is None:
                    rd = dict(l.split(': ') for l in open(f'/proc/{WATCH["proc"].pid}/io').read().splitlines())
                    io = f'flare rchar {int(rd["rchar"]) / 1e9:.1f} GB read_bytes {int(rd["read_bytes"]) / 1e9:.1f} GB; '
            except Exception:
                pass
            ra = ''
            try:
                ra = (OUT / 'flare_ra.log').read_text().splitlines()[-1]
            except Exception:
                pass
            log(f'mem {m:.1f} GiB (peak {WATCH["peak_gib"]:.1f}); {io}ra {ra}; server tail: {tail}')
        time.sleep(2)
threading.Thread(target=watchdog, daemon=True).start()

def one(pattern):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    assert len(hits) == 1, (pattern, hits)
    return hits[0]

# ---- 1. locate inputs ---------------------------------------------------------------------------
BUNDLE = os.path.dirname(one('ld-linux-x86-64.so.2'))
DATA = Path(os.path.dirname(one('replay.py')))
DSET = Path(one('qwen3.8-flash-next-tp1-c16-mtp5.safetensors'))   # route A: the gcsfuse dataset file
assert DSET.stat().st_size == TOTAL, DSET.stat().st_size
PARTS = []                                                         # route B: six notebook-output parts, staged
for i in range(6):                                                 # onto local ext4 before the notebook starts
    meta = json.load(open(one(f'flare-part-{i}.json')))
    PARTS.append((one(f'flare-part-{i}'), meta['start'], meta['end']))
assert PARTS[0][1] == 0 and PARTS[-1][2] == TOTAL and all(a[2] == b[1] for a, b in zip(PARTS, PARTS[1:]))
assert all(os.path.getsize(path) == e - s for path, s, e in PARTS)
RESULT['inputs'] = {'bundle': BUNDLE, 'replay_data': str(DATA), 'dataset_file': str(DSET), 'parts': PARTS,
                    'mounts': [l for l in open('/proc/mounts').read().splitlines() if '/kaggle/input' in l][:14],
                    'replay_manifest_sha256': hashlib.sha256((DATA / 'manifest.json').read_bytes()).hexdigest()}
log(f'inputs located: {RESULT["inputs"]}')

# ---- 2. stage the binary behind the trixie loader ---------------------------------------------
W = Path('/tmp/flare'); (W / 'libs').mkdir(parents=True, exist_ok=True)
sums = dict(reversed(l.split('  ', 1)) for l in open(f'{BUNDLE}/SHA256SUMS').read().splitlines())
for name, digest in sums.items():
    if name == 'blob.bin':
        continue
    dst = W / ('flare' if name == 'flare' else f'libs/{name}')
    shutil.copy(f'{BUNDLE}/{name}', dst)
    assert hashlib.sha256(dst.read_bytes()).hexdigest() == digest, name
os.chmod(W / 'flare', 0o755); os.chmod(W / 'libs/ld-linux-x86-64.so.2', 0o755)
import base64, zlib
RA_SO_B64 = 'eNrtfX18FEW2aPdkBgYIMwEJGwU0soMEbwhJIJgIkQxMoAMTiHxGRCchmZBIPmaTSQiCGJyEpRkGo6Kiqyt673VZr7uLq9fLT10MBJKg8pUVZf0iomCPAUFY+RKYe05V9Ux3MwP79t333h8vzW+oPqdOnTp16lTVqY90PZptn6rjeU5+orh7uBDEcVksXCsqcelcP/h/KDeE0Oq5yM+FGHXIsRDTGfAlnuE14epx6lCZjuRXwPCacGs/dahM1wt+XRUU7npEHdqiWDkX8Kp0OpbOWElh4yp12MKrQyPLVs9+W3QU1oYWTh3qNekallJYGyZz6lDOb84xd/E/ky6PpVtbxMqvCcfx6lCW815I14v7xx+5+mez/CLVQ7xOHfIKvQwi9slx02bO41KPZGe86dfP+OOgx/ps/uusk8Mu/eZWpBsAvztYvfWmFajnFnJxYwftND1z+Im+GDcSfnfCLxF+SYz/GI1+UuA3FssNvzSGy2Dh6N+eGvj8x9Emm04YuuzT36ysd05I/PSnqTHzu2b5/rQr1/xdS2nsr954fuW7hxp25p0bFXPxy4adS1Ozht5nuOPXC5+fYC7YdGrggRdfGFj16tEoa+6LK6+ntypQwMAw+C4uPP5yBHyMookpn2MR6Dsj4P8jAj49gpzOCPR1EfCvReDzSgT8oxH4ZEegfyQC/sEIfEZFoOcj6DMrAv28CPzvi0C/NgL9XyLgo3Th8U9F4N8YAV8QAf/LCPiUCPhBEfCJEfRmikBfFwE/KQJ+KvyGh8FviqC3/hH47CR4GESmUPgh1hHdraP403OYHhl9EsNnzaZwLKO3MD4tWWr8BYbvmkFhA+PzNcM35FD4G4bfw/Aum1qeMoZvzmT9Ah8KCf00CsvDdxqTs0AjfyejT55K4ScY/jOGP6DBn5b5T6fwYpZvX8Y/fpa6vL+U9XMvha/Ifaqsh2wKs2TcuzLequbznVwvgrpct8j889T8P5f1I6j1yTkcSyqqKh017sJqt8PBOXLm5jqKndXOJWU1bmf13Nwp5VWVzrmFi8udNC58jKOovtBRUlZZWF72sJOrdhaWuwrdpVyNu7qowgXRwL5oqaOodKmjpLCsnKupdFWXVbpLkMBdtdRRDSRlNVVFqWMdBFPOudylwKXYUVHrdtY7yquKlgZRRVWVxY5lhWVuDVFtJSErwv8dS5xud1mFkyupdjrVKRdXVxUWFxXWuCFPZ3V1ZRVyL3SXVVVytTXlTqeLc1XVlNU7KpwVUJollVxJlcsJ/zOJS5ZVl7mBL2RTAwEUzK3mXwNpCss5SF7kWs5VuKqr3M4iN1dcXrO8ggOxnJV1ANQhVFRYDnmHksP/7pC0xU5QGtFhdVHpNRoq51Cs8eNIADLQAGAXJpZD+n8dC2RsXWoQAa8VVIqiKtdyqMJyp6O6sHKJk6txVhYjGHyB1DWu8rIiJ1dRUegi/wGqvGxxUVJNVdJ4bpo9Z/IUR2rS2HHB15TQ69jQm+I1PfiqSHRXKD4pNfiempRG3Bgd8WmiwJ2JgjcD/HoRXFQwrjd4dX0Yti/80wVjEO7NIP4f/CePc304qy7kv731xDPg9vXlZjDcRgL34eYz+Kaysv7oXRYy+E0S35tbyuBtBO7F1TF4EKE3cKsZ/CyJ13NeBlffXNYHS/0Mg2tvQVjHvaJT+42yP7t5JA0lDb6BDQqnNfg8Rn9Bgy+QB5FVaryL4eNWq/Ey3PEoDXtp5kUHFHjleHNYgb9N6d8p8L9UDkwNNOit8I2JHAq8Tun3KfBRCnycAm9Q4OMVeKV/n6DA91bgkxV4o9IfVOD7KP0jBb6vAi8o8P0U+DwFPlqBz1fg+yv9FwXepMCXKvBmBd6lwMco8PUK/ACl+hX4m5T+mgI/SIFvVuBjlf6IAj9Ygd+swP9Cgd+iwMcp8FsV+JsV+G0K/GgFvkWBv0WB71DgU5V2q8APUdqtAj9UabcK/DAFXlLgb1XgTyvwSr/wggJ/u9L+V4fwSj/PqMBblPavwI/gep6ep+fpeXqenqfn6Xl6np6n5+l5ep6e5//P56z51kuC54RR8Bp2OGAZpqnFrQscEDy7jK0kPpD2AaADIz6C/823ZcEbwqUY5e8KwDNiJ8K4FOU/QOB3EMYlKH8Lgf+MMC49+bcS+DWEcWnMv5nAryCMS1D+ZgI/jzAuPfkbCPwkwjh997sIvA5hXGryFxD4MYRxicmfR+CHEcalJX8WgasRxiUlfzKBH0IYl5L88QRejDAuIfljCLwQYVw68nMEno0wLhn5T19FeDrCMaT8BJ6M8ABSfgLfjfBAUn4CpyJ8Eyk/ge9EeBApP4GHIxxLyk/gWxAeTMpP4IEI/4KUn8B9EY4j5SewDuGbSfkJfAm2MktvIeUn8BmEh5DyE/h7hIeS8hP4KMLDSPkJ/BnCt5LyE7gT4dtI+a8g/AHC8aT8BN6J8O2k/AR+B+HhpPwAew3P/5nnmlpq+Vb/AdCi1/AaxDe1PApwPNSCdYF1vnXuvDnC6hMHdDxXLHj1IxIgsSA2WjoAkdIieDI3AAvO3HgcrErwZSbcA8tJbuCNxmpoSOdqRwpeSl0s/AtNbm66ACYnrN6FTO9/wLrI+qDV0Sr40qJhI1kQ2yXbVZp+t+BzW+KB69mXeU66FbAg61bBayPIDzNREkP7ZggAlQDvb7L3PHh/Cd6l7y9TTn1AEnOTncjY/9+Bm+CLfWUix23HdUUohi+z+WWOs/syfYCUxoH8KS3tBtdmuo4L3H4knA3TNxOoaDMp7CKQQPo1ZiG2bwM6vdAUcMe8g/adck76LWphR9cgIMymouoh2btondJsJtdARoylrkR6EfSQOYKQD9v9ElAmMPaYDNlHy+wnIbkvsy9lLSLt5Z8pbbKW9iZCOydK8LTqIc03UEghY6dgzm2FpPMx6Q5M6uu/AGLs4iWiFpAPaPNhjV0wZ3ektEjDUBRv/3uAfjvaOOjj7pdQETZjQHhOOH8RpRfbgeVNyPIeIN9uJOo7Argjv8W33dJIUiBDB4JNJ82NX1JjMTd+DC92X7El5n3UuV08LpWDUCXmUy1nBPOdNkscvO45Y77PkAs2hii7JQbtDxhxZ8z5O86YBxg+g/Maq1JTPi8x3wZ7sSl7QLO+WIMe2X0nPUDkX4mpWnlpDEIZw/Ig+aOfpOwBuly0hXNIOVamlPrAW7uh6k9008WfDDJKh27G9khUGvvA3ViqHSktKSe3N/BoV+9Z8JhBjvipdPISmFEn2HXqBC4YIYhzQQAf/CdduUQq3L8fOG/PooqSDl9i6tgNOdu8aadfhHbouWo2P/4WINqQ0VuWrbJWbRaj1OtnbBfmpo1E6LkWIxb6YZ7UvbQc2WXsNjfVQWxO0VXBZ3iVVMTfoJqFHd9F06rm26F+M5G2KHbVi6SVBCBIgSKmuUkREw8K4gEh6nVLF0eiC16kWE8LT0ojGqYCRhpFyyQ0fW5uSghg3qMHv4hN/hxI6e8AIaS2i1irWD9gXtaFGBjuxiDjgjsWbGpfPh3q+gS6zLeRleZWFgL9AEKf1heDUVcF8bSw44dJwo4LUQLfJhy86h4EDF5kDIyBLpqPnB7H04bMY/fBa+3YeWA7D+cTRR5z9xe8ma8CPhBbCiipCCRvM7wHCP6B1pIk8210J74VB9mSEIz8/EuANoQKvZlvgy7uRBxoV9VvGkmH4zXE/wF7XrcB2mntSOiBvBtJFHSSgwhtbTp0kJhakb8AbSLhIu05PgItfPEC9pffSd9eIBawscX8dkv3f0pOAKl+V59I4DX5x/GYf0PmS+gmmBvfwFX7zMcBgA1XgNcTshcs8Typ5YdLMYhddhds5v+8nSNIbBYDL6D1r6SWbBTawQ5pBaXNAlLplxdIRwRy1t7RnVMiy4/mhOS+tFFIdeW8TOUekXKO2msZpMxorT0keKkM27DZdccwe2FIqq44XqkuI6gLC9sqSy89eB7VoKwvMp7NIyOa50S9Qi02S2lKS0kz2Jf/OUi56wLp6GKfRzvaUzsyqH9f2p7niBlORrIvzxGyH54jZO6vwfS+x3dvbDcGHj8viP0PwKvdm7hTwIYCnWTmeUJiQPwMX3kFJ+w4BsYrCWLsHwDlSefdAyCDQsygHEpA7KudbBsCejqi56HevGlvwzvpKoHbnzEX859+EMSd033RSfZR39vFcqidjB2CeeYnOIzmZacEUj7H3scoDTpH+1XSoZK+NR/71Z/S5X4Vxs9ySzJWK9nOYr2rzZIueFYCytw4HQbwkmab13AvZJydci4bzG+DC5A2caUlMVc8D7juATmeNt6WkTYaaMwbTmGnPmGlxWj2fMvRYTom15fYG1r/RiCYmZF4+tF9gm9YbwqcXbXDLp6HSNTXdty/xN6KdFSB2ClwEk1shzzXekAx0736s2ToQb8A2Brto7rtoy4IGa2CedZh7IoaT+hwCFhpibeLV7JTvrWKYImGYag00S+In9nEwPR/iT6NiePso47a+a8F37iEQOybv+exJzFvOINdOr+LuB45vkR9TsYX1X+BVFjCHPFUILYQKKGok+Eok/nx2ABaXhaXK161wSiSmPK5PcMNBd/Yam76AtuQF7Xry1z5LNTmArQirCBSl1L935VjXr485t13V7BuaG0sgtrACAEiSsxjSGj2DNURssQWu3d05bPYsRtmwWk+O38StTBMR/UeZ/emFUBsezbZdhY8BisQgbVmx4AtmTAdWquYbYSBOvAMsTAOsb5SnigY2MbVYcMZ7YdYe8ZP5g1vkzoddjOSQV4bXiVjuCS070BHXWhvwyClsztO0R4Fz04ecooTMj5zj/WkR9X+4K3XQ3/4neCtNwoZ5wXzZPx9oKCrPaxKfzHgzhJ8+hzBc5LP9RXv5nLIqB11FrL0GGbUQqlywIrJ5rG1XW+hvejosSB1TsYP5vULA8SJevNp0PwPZ6C1rb5MBifK322S9gDSk643N/YF7xMqPBC77Hc8pP3C/Ni/cjjMn/IPBk8gVNM28eL7Wf2jYMxu5//6nSRCdVo9gV7mxt7AgJpB42X0HWhLatpGRu1Y/9PMGcVeVZB0f6djeidSgsYFaGrHnyZNDc08Dtvb482XKRsjV/trYlLvEh/lX9HTjz3wKs/5VxAfAuTKEX/w1wEe7MZwKxxZEbazKWYDsRYh5bT/gyvU54mTdv8IPU42OSEBunl2I7GQ/TWgS+kDiPI/8LM8vnhO5APFvCIyxoxjnanbEgcdpR0O6UpDzjJn5iMycmR2z0aCReCr+4ZNmoPv2BDE3Xax2JIO7RI7rSy7b3RUAW2ueeCRn1+I725QymjkJh6n7STvx1AfJoinaHMxYhBtIYMGyPBBMvZd9xmGjpP7sEWGbdWQuBUS05EliY4sCqNSvOZ4doHdtcbneC5nmRt/D5hZXtsxLhcGrG+B/QP3wwlE78TfNbXYxK+ne8cdh5Ey5wHiz8yAnGd6o48uAFRJMhb01HzBbDsAqdpTcACz+2q6pIkgRv5CxfhkfnswUNxJE6S0YO8i/QKIcjK+qj0qiP53dAqpttNSZuKxaGnHjzhlIYI2LaBG3T0G2fjfwSZOaKUXfiQRTjjmJ8X8SC0rLjgtq31A8ELV2L1QF5C9HSwqi5iV3Qvat4Np5MHSQ77/3KlrUhowopXWLFYpqU9amVhzpCqlfaeuGY/n5ogfzxN/wuE4JhAb+ytSnjY4JSi2SpNAQPF70OU5UKnn6CTBoz/AYy/2G+qAHrkfOxz9QcHngq5q3EF0NiqBvB0cOHFoc27Tt+bGN9FJs5qr9BZwXsVTZPSDKZkBkm+GXMyj9DxGQwdhHmXjY6QPzzEXGtve41S9dwGhdBRkl/qTZrsrRjyMk9bWkmbGUdoYSlY7BJLoMcmfTqmocTwA6UbdR5gehaOUMBpjkaQ1WkJO5jtRIU7jr0itGqR8wuBlBYNsFM4FTb17m1q/4P9tooV40qVpotBu034/GsebU6yJpkYRSqmZNtFSMqKWW1xYl/V28XVLA+nR37KsJeF7lmY0qedO0dEri7bKd06ixIbWKhyU5EFMwDa4KZmg9sDA9EiVPJZlCSJUjdfwmyp0kkYsqsIqHZ1bRX0yXPcqJDHDUkkQ+0sWg8PUrCraqMFz15PXRYYvKml0HETHE15pXyPK27+TxYB/YBhPeGX+ggRpyc00JgFiEkma2NhmMn8wr13PE5k/R2EPnzHH9P98DBkLo2nxOu1eQx+S9YN/QxbJ2DGMtXv7oxy5TXvMa5+iDDrP5JjvPAhkB3OR7TyK3YPYDsRSvfTfXUmG8h3mNVE8Y/ABFyIFP+DB/mVjSH4d1nnb9KQFzbPOsc5eAJPB+VCjH0KN3mtdEIA2B798+JXCrz5AKw308BBkARBahXX+bChrIUXg6oX0hxNQeat3NVOTObIB109O0Laer1i80TMPxoU9RT3kbPdS48Duo9TuZRbipRYCDDeR9aZuGP7EDtJdH5RQXdBBdHi6eKEPejLEHMChyd/jR++J9cpA7B9InCkyf8sRWyAB1kjOQSmnTwtqBeol/3O/AwbXkuZcEXxGMiLkHPQDZU6fHUxxwPapq0E+kLHV8zUy6gRZ7H06MPsDQNXpX4BLT9T8b2j7V77XKqd2Fytuqzw+GpsV7fF9MsUZDzqVCrG5ntLMJ6G9YkV4DVEvkdW2Qay5brRsgVC8hHPa1ZcCsFJXZ8FVNcQqKgb/VgXm6Kt3baMScOKn2DRjoC+JlrpA/9CVREMXBW6s4QkfTrihJdW+DdAKBsXVbgEGvv5FK4kJfAGzwTY9z0tvdpOB46P1xBX4eCm6Avuh9NLzP1Ou/gqoJemEPxBoVvlnJ8A+wEERxEEW1XQrHaZwTZ3mtXP1pEGdRcbeaIuQ0WHeUIL21XTO3PgAvozqwOoQhIw2rJE8mM60Ya3kg08NcAGOL6U4vrhov5BJtDOAeEIABEAvv95EezbTevSfTkrHvycd6fsPkcI4sDDQkLNI2fSHeaX8Xuyd+mdDVI6v3ACSS2QKNxFpfUOzcsRdwo5vokDqR0zW8/q4n9FUG79CL9uT2Qun0uYNWBe5UAu8lfSQ3fEq/6KdxzHenpH5Ny8Qr/eSTDM/9qJ7evWRIzN8+jpg3IKMaw+CA0v+kg2kfdhL9LbCi749DPs4t7TDuiSiz2MPXOYlZX5+HdrruAPSyxI0P69hIaC71wbn4y04uqQHJ9Hmps+J7Lvqqf3m+ibGWT2TosAPpdPMs8gOvAGY9me+vw4XpqbpeTpXMHy6DhvoNOy0h3rJbLTLPupYrvj33FEnhFFSTsYuu3lal7RJoo3GaBePTvdNxG53N6TM5Y8JXndcjMD/IM2TiLU9CGgpw0/kfoVkVh6n53NhsvFY0VVCMRHF4Uc/XQaEWVhAqB8vGgv0TnnYO+WjE1OAEpei/+LyR6sWa8jyMh24+O+oO2n19g7NihNw4DoySh6t4sFcYiAv3iNFBafEcZ4LfF2SvJ6V9nuR6H0xBNJfjqPsmcdFnDseAc6QDqiXHVbkKx5XTvcS5OneQ6FMg9O9ODIPHEWnexjWjKTOO2+oKUX1TNK7x0F+dTQ/v/2q3I/i2Jgwis0MBfM7FyZBmkksTS2sJGWeWYv9GcgiHT4Gs5mmk+3k4LHQTraJcszZu/2vhPpPqDNeJBOqOOaDpq4ljenoWpxG+mw8Ww/2Gr5eiwZoOQDLG/swj5uJTgyd8O7/FPrAd3WsoZo585oFiIhiiEmAmIoIPUMEAJGGCIOMiDKvsQAC/CCEzOZnWz2tvfwDcJFxwDFcNRkcKoN/wBXaswukHyF9iKYHkRKPXdN/EX82BnqjDeh2iueQgzHlpCD+LOw4rkcuceDDIqN4uy/xru7BxM/jdwsTphirZ4GbHmPP6HZPFbw10DAm5PgGJUrH6Qrv8r/B8jeaqxEtNY6JCZMECY02HvsaLy7heHYlBNc/g9TXkqSc7D4F8oLK2zbx0EjnWvJwDdW6+gLpEaOBSLAGjlI7RUfdJsIkZNjyOjYLQzWkoxqyYBaij2psqb2dTAeCM4B0bEFZjA21q3M7dW7DuR187SG7uNcKo414Vlh9FQSNWbbS8/UX2Sl7rNuwUGQK4tkz3BrozBHbBM+FgHuiAL69+BXue3xyEWfWlwCCBR/p8LekaReuAWOZeIy8z1tDjKvWicNO/rfYzk9Hko362SJZmcwcugYXpGyWoaIdVEZcaj9lr6MsbyEsDd/iLBnIckQ/COK1owOJHEAi9L79Z6/S9RxYN4R6R6PwGgqKiUedyIa0ckgj/kQc6Tt4tjj/A+zViq1ew7wpuKlWb4X6GAprI+5fQD+xbjiZ40mjviET/cYdbEksTjGe48Yt1q2AMwMcz3E2AumkQUeZz45LELiz1fg8EWIHzDY/ul25q/UOTLLbDQcm030s6f6jpPQBoJH2f012fDpuJyu8T5NkvLzr89js4K7PyKPyRDvJsAJm+g/DWsZTxEt5BNj6RxL3TF6fHnZvEbIbNhzXUhIDbJ2g3TBzMl1u8McFFOsrkO90kn3aTAza29FYpDu+IRtEgcXUC2/aSjr72BRKaVFQXj5KKA8BpX/dVcXiwNivlb2qUe5VB1lorxptkfeLXsOkqcgrGdWBe+2wDanI4maisbSr8UEUZLjradiODBD9pXWrYl7BmGM05rAqpgFjPqQx7aqYBzHmDRrzR1XMRIx5lsY8p4oZiDGP0Jg1qpjTGyGmiMaUqGL2YUwOjblfFfM6xoyhMdNUMeswJpbGjFXFlGLMz13kfT28k1MJGn+WNZcCnM1mqfy/BLJ1PGcj2To+hK5I5/m2LM7K1U4Ah6sY8Ofb8PMGtfdDS4IV34PQT3z1dzobTlD4K6+RXnpXwf1sfm2zuHC5D9f7gDKLTgYBmwxcTSglzgEpa/e/A2vo9KLJooAYbZFyj2MG0RY644ZuJdq9GtrCJ/BXCtIbXaTlfLCa9BsfFGC/sfcIOL9AhfSYbvXxwPtg9tOD8xdxnEXagvvs4yz+IWhNH34FUf+4fsY+FV4/E58Kp5/tZ//39HP8yRvoZ/SxsPp5G8Ym6YkjRD+vNRD94DESXvrDV/+AftZeVujnP75U6QfmQzjhsoqws2y3lKo0JBANDXiSaOgm3BTac34nasjc9CX1kU1YnqZz7oyGe8a4x9nE/Skt1sABUNTTZ6iiBIWiztGpKmbXGuyfzrVluWE7+7wNck8ANVjsqC8R951BMUZpKA4mUEIjDBkxTbDmDrHA12IDOMHtQ//7UbrckoAsLMAiDlnEAGCUHvyKqGzZo0Rlyx5ElT38JZlPxSHXBGRjQaYx8GJsOP6+/y7So0I9NpIZKFRlMm7QQHQpmVVhVebhW5b/Z9IfwlosVh+Oh9KtP1Np45BvDBnYOoBIGv9FcD/2hvp+uTmyvp9pjqDvET/+D+n79aM30vedqyLr+4cviL57ryL67v0A6rvvF//n9P3Hi+H13fZZUN+eE3GgVPvjoE536uoA8dT6QnuNC4yIAaR4Aca9T8txEq7jKPriBp4L6guyiZamQTbgrUR3v99MXBNP5uUNxAM5LYifuBO9sW8D2NQJKweU/3RBPEO9ijY4P4TsFPvL6Prgyhtw1pOTCpJE9kLtVEUxZOYGuem7f4frC5fsPsNd5WThelBMY6f73wTxA9zRBfb+wCnqNGn9aU/mIrV8g68j3903lE+4kXzPLg0vnyOifO/5qHwg3ghv7DKfSrwpCukaIYr6mwq5cLllkPTGeSYWSDVIliqmeyvt91CuwEMhuf5DIdevz4bmWVSeRJU8362PKM+J9RHluf2G8lRFkOf8Ga08L6zn6RzCCMMX2WYWf5JnQu4JXkP2enK0wbj6ZyJhBsz0wRmmsxycr4BvjBMZ5B4YIRCZqf6lFbDg3b09aN9kWiXCdDm2E6bdMC3RwbSkL5n7dL8G9fsOT443iUcCsX/z4uB1Ck4NwRZ4DhwzasCVaHE3eJCrzDk+twFGUKR7g9CxKRaRAiWisuE0qqT5Wnt43RuhvFZa3nledXlt1ytvlldRXu9PEcp7tPQG5fWvI+W1BcuL+2Fhyrtj3f9aedPR/tex8sbR8saz8iZgeacAQdRduDm60Gu4f52q5I+GSk7mZvG05JhTemDEvvG4dIH2hnOyaOJSrPs7nXpFk9U/T+bvx+Mq6qu4fzg46G2GyL2fKMlFwwog99cG/QmcwyYT/b2yRKk/wPoL0W3F/eI7CUzkIto7FRIXrR5E9etP065Boe/lokrfbVLXqaC+TTm+cb2oumeLTN1yFoQvzQE4t6r3Z/yfH1L7O54TpaCEjWkcLUu+VbyKU/KCHHG/QHfTinH9gq7pJthh88TcuIwuNy5II+d5nqPHMIcvw/WcQ2TAG7yMDHh3zMcB7+ZDdPkthswfh5ADAfJmy0QLHEYyN7bhXyjj37ILGe2CeWo7OdIHI6108mO5yObGd5i55tlAmkQ0lnRYhsF+JUf8AlZC4oQJQy1mzz06eRXVjUMxTPlPCaMOCxmnBfM0WNBtI/6m9DEs8uHCAJ60iNaRVdoc8YRdPAPnWHLhHOOSHJ99GAcdVkJOlBmaeCD2oXFkcrjqkRx+N64Ien4k8pyZ6SsezUGq7r1Yf2QyCNvK0tiQ5E3nseGd6cTlIXLQ+AlyVov0b5gDYX8TY3+rzP5JDftDdPwG2+8rvdpJjgi478P5vNiJhwGkHP6wOwUNIB1bXl6uuMsudqEtZKEtCGgf+bhmUQBWUdoaWm+WE+S070LL795D12FxzfPk1WvGKzt/1j4BlhLNnsfQ++AXEe3bvFgjQVZK+wIBpYNKAxBJrnZ08I25WPnzINdcfqQ/F8JmZv828Xwgtn4NnR0Y/X+9Ejq/l/aHgaG58slObGKZe1Ppkdl9qejoVV1RrA/40jwK+r90ktnjb4HOL1xhrl1KJ8muD2Tnfwgbf6g0YG5GUiT/WOaX0f4qGmjBab2UI551j0P/AgVwV3sN7zWR7qm/lW6amNe8S6rxKxssVZEuaWwqOaeHzmKinWwP4siJbYs2MvSvvjuFQiABOXfsybySwpSGvidsly+yxNhwD6R2A5Tv9QGKbusarnYRicl+TMtBFVuZHeojBd1pmas/LbT/ZcPNmbS2xaRv69UI9TcmgKthcoH87/6s7U+gz75kFX/MEf8Kkn+DH5ZyZ8Db7VgGd7HX8FyjrKKLVEWbiIrgRNQ+yhSGrJ+SUUu2oP+Fpz1xXggbO9KeH5hfQSQGzu14+BgLFo3iG2tfAIl/F6NYOSBLcUQNSEMW5d46oGIiGjzJSh7+eew8lU3sgL69rVDu282Ns3F1WuywiZ0pe4ge9hOf5c1Lsn2oy++Ry39zslx+z43Lf3LMdcp/UlP+98dcW/6t5huUf7+m/CvGXK/8BTcu/0Vl+zjxmLp9DBrD2scLj92gfXQn3bh9tJ/QtI//TLpO+9hq+gfbx+Z9kdrH/KTrt49XHRHbx28uaNtHOizlXhmNO1eoHuzQb8FVQDsuRn+aJ70mkbMZAtvPFcnWV9qZ/hxH9zVpSew4X0UvxQIv8SD+UOnrvbjPgWgQ3wIv8SDsUJC+eTSTniwtdz/ZfK08zsjy3BJWniU3lufeSPJcSFTLE/ybo+BfGfU8PU/P0/P0PD1Pz9Pz9Dw9T8/zf+PBP8kbUQOfZU6M5wq5FcPdw++OH5GUUpIYP7zQ7a6uAXAFNyI5tZ4b/gj3yCN9lV87pZ92VX6ndKrdOjvbMWWewz5rWgiYMyMnjyuqnVZd6Cqd4ayudJbPrCp2znG6rZBD2WL4iC35iut1P+x6zQdd6Xdoy2uczqXBT7+OcVe4xpSUF1bjt1yTyquWMBlmWx151rlCCJoza97sKdlzuAkh1GT7rCkzHLmTQxirkG21hcC5wmyA54QQU6x5IQALLH9jln4p10UClPSf+8AsfUbUDB9RjJUwHD+ai9UxorwWKgc/JxwEapaWuVzO4lBkWXWN21FdhIjiIFxXWF7rdJQ66wEd/HSiqsbVWSxe7nYqMnS6i0q1sCMMDejQVeMorFsS4kvxNQ6Xs9qxGD9TTKLGYhR+zzjEACFHTShhtbOmrNhZSUQsx5L8qtZZywqKoLOurMitLDhoRQGiCYWYF5dVwweJ8TPCam2hvh01+DVgNQrsaIkziCK1qFZQKXwQGoD7EbrO7wFoNdzTn9wh9UsacA/q3Ll3L/uqdRf5unhDY5KVHxI1Ae9YwG9UbnovEMiD98Pb4W934GiiBGEXfAHjAoRvAZwOC7f1+IerEOLxlo7dgQCe2o37MBD4O4T5MBHPgzXubbC+8W8QpuMiKJyPK4VwEYTpf4X0EF74FPhCuOVwIBAHfI2wa1yO+X0eCGzCfOCkwGUI8+EUQTyciKyHAxg2CCU4qFIAYT7sla+EcAuEL0AowKmq9zAewi+QHs4uXIbQCIc6hsIXPS5AaIdQgkMwKyHMh5WGbRCWQvgRhJvgCHg8lHMbLPdkYQgLXwUYwgJgA4SHYZl0M4SlsHfSwj5CK39rlX94NsfXx/BDonsb8QCBhX2XNOtd4IUHQE0xU01x0839lsGXcifdMuHOsRbSBnCvHT+pvm1nILACEdNMm+NX9TUZZ/TjEti1IkNbAwEdT+K26nRTPTpdXWuuRye0LgI6az8B/0AUF/vwNDYHdTIDT49ONsU8rptsitsQNdkU79NPNiWsN1hNyWt6WU3pnt4201aew78Mb0H+ewKBVJTRZiqo8/S2r+klrIe/H9LbH9fZN0Tp/tNUOsPkWmAqtZoKdI+3Zn8k7M1uy96Xvd9mMmabCuYSIR709F7Ta73Bp98Q9Tiyws+gD0WZwSZm81Se9TqbKW5N1BRTvEdvMyXrXutrip9iirOaYiabjFP6OZVgdr8HlaANP6CyGb+XC7yiPwJdRVFd8SCwM5Rz1AretIUHPfFW0zZ+sqkFfh3wfoC3mQ7zgqkLYAneT8PvAvwaQKS1ummmZt1M0yZQ12ad1bQF3rfq7MDb2bct+6MZe7NbhX3C/mkolO53rW0f7d23f4myuNkUt8BkZN+UBgVIYPvku7E2UwzWcQLg9LAGuovWcQuvy+6LxSpGEmwr+2nbAvoGXpdP2S9ihUrgaQ4LTcapEK3MW7etbyvUxkfZe7FGZvWL+ozR1qorBPvz1/Eb1rBNtJXKlTxtjX6Wx2BfHyX4dMuBzUd5e+0mo9AvuxX52Um9wscyuG8h3VpIt4gn6fLQ7ASoccMawnwmkYDaQ04/XYMCzO6H+rBAXdW/EwjI3zdGu04H3EbYqCP3BUAdrwHFx3mippmydJV9TXGgNSupddRdMf5xA/QVVpp/TNREHlVN2k0DxE0E3gv1Wptfr7eaEtagzXt6TTM1rNHr5vU1JQPOaooHmsnErqz92Hd/O4DPOOiHYpk8HjAKl25xXyJHdj9CcwJokv8WmQZljQE5EqEfe4i212ZeVwXamt4P4xIhruHza+OwnxAg7mM4qTOL1k38fIiZ2u9e+H8WiccvFumhP3yC1V0ZxMzpB1ZqpP3MWohvvE78FkwP/ajI4pdAfK4ivgPi3RHi8c4MCeK/hfhvWHyuxwDmI6yPqgbCmf3m+XTz++7N7vfo3jkmI9LHQR/0b0f+Mfp0LD/QF0O/vpjWcVaQfiXQ2/vpHtgACfZl95vRd98iNLNV++YQG0CbaoS0m+AAp/wdcex7NgGwEXCjowg/V+6GKOFx3UJsEjqvT+/prXuyda/L09un181tzd77IPA0ERCMf+9MfKmHF937+Da9dfpe1NEJ4On6ltU/yLiiLzH4DlmH0TCGuGCMucLKrLOSfly3BQgwPhHi845HjsexKfm7yPzLMR7GqkaaPmu+SZhmysvBTiEL49dD/OYI8ain1/GrVv8Vaocc+0Z8PPsGO95rBB+BadErvpmuYzDfM1Xpef6JR74PQb7/QL7vQL7fQL7PQLYvK7s8IPit/XgayN/Yd8+lofytefn+hSGae77kb8vL9y3I34iPY+FPVwNVGB7Qqe+/uWDkgvce4LOWXVog3xXwR0bfT8NPvttAejR0bxvZ74pS+4nMbQx+W39rfzW+JVp9r8NpFvbR5AeHIYj88Yw+wGBZj6cZ/CoT9CKDe/0/sgP5vjrtM57V71QWzmdhCQvrWLiGhc+w8FUWvs3CNhZ+wsLjLDzHwl7sI/6DWXgHC8ezcCoL57OwhIV1LFzDwmdY+CoL32ZhGws/YeFxFp5jYS9m54NZeAcLx7NwKgvna+73wL3OaVOm3B2fAL3zqPiU8bDOkRKfmpw6Pjk9JTmyvuk9hT8GtPhoYptwHxRrhyu4UPvh0SoP8ap79obL+N3qewlHMT5ZCczeWXS6TP8Ep2rYWYw+bxSFZ7AGF81T+oLWkG9IXGTGZ1Oruj3kMnzXTk51X948ht+soV/E8Fs0+MUMf0CDL2L4rRr8Qwx/WIN3yfJo8MsY/kKr+t7ClQzfsEt1nQq5B4voTYNfH9Q/p1LoJVlvrL7kfJ9m9Oka/EsMH6PB/07W8ytq+d9k+BZNuf5L1o+GfifDr9XQtzN8swZ/gOG3afgckek1eInh41vV9f4jwxdo6C8ye9ucROFUXchOiT6fVA8YfRi+ZbXaDs0y/WPqiyuHyvjHOVWHP0LGr1PbfwrDx2jkHCfTe9V8Jsl4nxo/jafl6mIKmM34L2D0Rg3/JYw+K5W1dzYe1TD6rlfUeljB8A0aPiLDx7+ivpf1CYZfq6F/ieFLNfg/M7xLg3+X4es1+D0Mv1mD/5jht2jwX8r1qMEfZ/gODf4U008Lu2RTvuvlHMOfZvhdcnXqKJ/TGnvuzfCSBt+f4bv+rNZzrI7lO57Cb7EO4lZG36HhY2H0DROY38CF7v1DfPJE1t/J96VGUbzsZ8l+0XjGn9PowUbwg7n41er+ajrj75rE/DPG/z7GJ1/Tnyxm+DwNfinDCxp8HcNnafCrGD5Zg29i+AQNfgPDx2vwzzF8nAb/MsMbNfhUPdMz09tCpojfM/ptmnr5M8Nv1tzHtZ3p7TS7X3E0a3cfMvpmDf0hhl+rwR9l+E0a/DmGb9Dgo6JYO9LgBzK8S4MfxvClGvwdDF+gwacwu3Jp7CojKry/kc3ou+5Q46cxfMxINX4hw+dp8C4mT9Zq9XxhJcEPDPr58vOvjM9mDZ8/EfoBwXmD/OyLIP9hQm/mVi691scKRz+W5BsdnKfIz/Eo2r5kv05+zhL8Tdf4x5eJvZiC9yDLj1lP+SRo+Awh+Njg/EN+Uog997nm0tO79OHvHZ2uD38v6LwI9CUR8B59+HtWX+bC37f5UgQ+O/Xh7/Ps1Ie/z/PLCHzidOHvET0XgX6QIcJ9sxHuLx0ZgX5yBPyCCPiaCPj7deHvI/2NLvx9rSN14e9f/XUE/i9EwL8XAV+hC38f7P4I9Ecj4M8ZwtvJ3N7h6Q29ItRvBHyDLvz9q3dGoM+OgC+JgF8eAf9Ur/D39Bbrwt/r26wLf6/s873C36+7OUK+7/QKfy/uzl7h7wd+lgt/7+6eCPy/ioC/HOGe4dMR6KN6h78vNzpCvY/sHd5OXopAb42AnxMBX9w7/L3Nj0ag/20EPBfc/i/iymocbtzDdTvIpbyFRe6yOidHUXAewVFTWlVbDhfXwu45ATFcVlW91FnN/cpRDgcSEHAU1QFUikcQyOYzV0l2sGu4xTV45S/sJ2NQWVNdxMHPUVVSQkLY2ichybnaIZ9QqHbQcwuwHU83xStr4DZcVxB21nDuMrjMV974DhGSnXYEyZ43XjVcvBxlQwK8WhgKQM8dwAuefICgohb/D27+I0A2sENYENqdLJcd9sRJDHJbCjcjwzW8cD4DuIJ8eLOxI4SkRxzwdmN3DQe5sUuPHaF9c4RwfzyIJRvsCGm24RFF9unxJbTJjhA5DVAZfCUvQTVQKsh7CVRXYQnczEzYQIUUO+vg/7LKKvgDBLh9GbKoLqQVwBWSepTBOnwjRy5I8chxFkdJJSoBLkWGKqgqr3MmueCC6KRkjrInVcgOqMgvBEf5VztYIPN3BM+r0ChE0TMrlfS245ogTWpSOuesqC2Hq5CJHrhqWqFwwzCQsMMkcMam2qE5UJJ0F+DksyRJ4xQA0I9FkBwugSuFq0lpk1LYC0QnU1uoLOOKqt2L4YrryjlJVVzoHmyHuwItthJ07nAUVzmWlFctRlW5q6rhjEdtPRxuqXCVw+XOxcAqLAXh7iisri5cDo3CXb0crqYurHA6imsrKpZDEgUEdQYVriQFoaAoKJLDMXW2NTfbkT3Thrd1O4prqhylhZXFeAm37b6Z1tycKYCFNTpHtsBIBdtsQMF93XKiafZZk612x6ypU+dkz3XMtU62ZwMWM2X3U2cpzzIpborGy7RVcdobtNWR17lKPPyN3FnKe58jHReiZ55CV3xnKe6NvuaKcBVHeme3pgDqm8mzQtdRy3eUa+jVl3BnKW6yZhd1q+jl+83VGr32unB1ycmF3Fo+cJG6ClUSVjzVZe7yPewh7uxa8izFBd2aIilLFO5+dVV+FddKqr69PEtxTI6d56Kn5dR3nivJ5NNj8lEwdoG6uuzY46gxpGNQ60w+X6a+z/1aGw1rofSKdqVg5K73LMUt5upb78NqIXR9vSqaXjSvQnFJkJ27cDGEUA8kLJXfKkEDSUsqa5MW15aVF48uK+YIVFoIH6NLKl5eiQolIYxIJKbOWV2DF9orAQfEVTvLC5GQvbnK3VwSafdJICb8T3rApOqq4kJ3IZfkLGW9UmlxdQhSiANVA0fZ3MspE9pjUSbyO+RVWFFWBIJUucl/NE/Kf3FNDZcEHWcFDvD/Q88Qzd3fW3Tq0HLN3Fn94DItfBmhSk4v78vJ4VuafTajJn0625PTafbt5NBlCOXLK+9N50LrmVGK9PI+oBz+UXft/oryEdgenEwm77vJoVsjv04TzmN7ejIs7+vJ4WYuvPxB/5XF6TT7iHJ4OoL+5PJXsvSTNfuScijvY0axe8S16euZXL00+7ZyOOQG9V+jSR8frw6zNPqP0YSrNOnldU85HJSqTq+dSzRq0svrN3LY/wbyr2Ppg2IWqEPZDrkI9vukJr287iOH4/jr579Zk76lWBP2un7+r2nab8NSdajdatSm/wu77z5Ksy/fVaGuJ2364Do/rm0p0svreacr1HYWKf0hVndRmv1mYyUrP6/Wm1FjP18x/cnpO9g6WkdVeHvR5v+NJr28Diex9Mn89dOf0KSX1wnjfqU5DxFB/2cYTk4vr5NaWPqEufx17f88y19bz3L6IVz4/k8Z6sKMC1ksfbM5dC6jIkz/0Uchu/I5wDZC3+h9/f53QIT0q9eGT6BN/985o8xy'
RA_SO_SHA = 'a9c676bf8d034f87c56037c68332762479c653a2d2e78d71e823735585a53fd9'
RA = W / 'libs/flare_ra.so'
RA.write_bytes(zlib.decompress(base64.b64decode(RA_SO_B64)))
assert hashlib.sha256(RA.read_bytes()).hexdigest() == RA_SO_SHA, 'flare_ra.so hash'
SHIM = [str(W / 'libs/ld-linux-x86-64.so.2'), '--library-path',
        ':'.join([str(W / 'libs'), '/usr/local/nvidia/lib64', '/usr/lib/x86_64-linux-gnu']),
        '--preload', str(RA), str(W / 'flare')]
RA_ENV = {'FLARE_RA_PATH': None, 'FLARE_RA_BLOCK_MB': '64', 'FLARE_RA_THREADS': '16', 'FLARE_RA_AHEAD': '24',
          'FLARE_RA_CAP': '64', 'FLARE_RA_LOG': str(OUT / 'flare_ra.log'),
          'FLARE_CU_LOG': str(OUT / 'flare_cu.log'), 'FLARE_CU_SKIP': 'all'}

# ---- 3. pick the read route: 20 s raw benchmark of the shim's fetch pattern on each -------------
def _bench(pick, seconds=20.0, threads=16, block=64 << 20):
    stop, got, lock, nxt = time.monotonic() + seconds, [0], threading.Lock(), [0]
    def w():
        buf = bytearray(block)
        while time.monotonic() < stop:
            with lock:
                k = nxt[0]; nxt[0] += 1
            path, off, n = pick(k, block)
            fd = os.open(path, os.O_RDONLY)
            try:
                r = os.preadv(fd, [memoryview(buf)[:n]], off)
            finally:
                os.close(fd)
            with lock:
                got[0] += r
    t = time.monotonic()
    th = [threading.Thread(target=w) for _ in range(threads)]
    [x.start() for x in th]; [x.join() for x in th]
    return round(got[0] / 1e6 / (time.monotonic() - t), 1)

def _spread(k, block):  # a different block each time, spread over the whole file
    return ((k * 997) % (TOTAL // block)) * block
def pick_a(k, block):
    return str(DSET), _spread(k, block), block
def pick_b(k, block):
    v = _spread(k, block)
    for path, s0, e0 in PARTS:
        if s0 <= v < e0:
            return path, v - s0, min(block, e0 - v)
bench = {'A_dataset_gcsfuse': _bench(pick_a), 'B_staged_parts': _bench(pick_b)}
ROUTE = 'B' if bench['B_staged_parts'] >= bench['A_dataset_gcsfuse'] else 'A'
if ROUTE == 'B':  # a sparse placeholder of the right size; flare_ra.so serves its bytes from the parts
    PKG = Path('/tmp/flare-pkg/qwen3.8-flash-next-tp1-c16-mtp5.safetensors')
    PKG.parent.mkdir(parents=True, exist_ok=True)
    with open(PKG, 'wb') as f:
        f.truncate(TOTAL)
    RA_ENV['FLARE_RA_SOURCES'] = ';'.join(f'{path}@{s0}' for path, s0, _ in PARTS)
else:
    PKG = DSET
RA_ENV['FLARE_RA_PATH'] = str(PKG)
VERIFY_WEIGHTS = ROUTE == 'B'  # route B serves a placeholder, so let Flare check every stored range
RESULT['reassembly'] = {'skipped': 'no local copy; flare_ra.so serves reads', 'mem_gib_before_start': round(mem_gib(), 2)}
RESULT['route'] = {'bench_MBps': bench, 'chosen': ROUTE, 'package_path': str(PKG), 'verify_weights': VERIFY_WEIGHTS}
log(f'route: {RESULT["route"]}; mem {mem_gib():.1f} GiB'); save()

# ---- 4. start Flare ----------------------------------------------------------------------------
STATE = Path('/tmp/flare-state'); STATE.mkdir(exist_ok=True)
FINGERPRINT = '937c0143f6856d3ee8ad164f62e3d804bd825d7a8750d0accb9a5849f703a8c1'  # sha256(publisher.json key)

def http(method, path, body=None, timeout=10):
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}{path}', method=method,
                                 data=None if body is None else json.dumps(body).encode(),
                                 headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read()
    except urllib.error.HTTPError as e:
        return e.code, e.read()

PREFIX_CACHE = '64G'  # stage 3: SGLang's reference ran with its radix cache (replay cache fraction ~0.92)

def start(fingerprint, prefix_cache=PREFIX_CACHE):
    cmd = SHIM + ['run', str(PKG), '--gpus', '0', '--max-concurrency', '16', '--mtp-draft-tokens', '5',
                  '--embedding-table-storage', 'ram', '--trust-store', str(STATE / 'trust.toml'),
                  '--host', '127.0.0.1', '--port', str(PORT), '--served-model-name', MODEL,
                  '--dangerously-approve-key', fingerprint, '--dangerously-accept-publisher', PUBLISHER]
    if prefix_cache:
        cmd += ['--prefix-cache-memory', prefix_cache]
    if VERIFY_WEIGHTS:
        cmd += ['--verify-weights']
    logf = open(OUT / 'flare-server.log', 'ab', buffering=0)
    logf.write(f'\n==== {RA_ENV} {" ".join(cmd)}\n'.encode())
    p = subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=logf, stderr=subprocess.STDOUT,
                         start_new_session=True, env={**os.environ, **RA_ENV})
    WATCH['proc'] = p
    return p

def wait_healthy(proc, limit_s):
    t = time.monotonic()
    while time.monotonic() - t < limit_s and time.monotonic() < DEADLINE:
        if proc.poll() is not None:
            return False, f'exited rc={proc.returncode}'
        for path in ('/v1/models', '/health'):
            try:
                if http('GET', path, timeout=5)[0] == 200:
                    return True, round(time.monotonic() - t, 1)
            except Exception:
                pass
        time.sleep(5)
    return False, 'health timeout'

def server_tail(n=4000):
    return (OUT / 'flare-server.log').read_bytes()[-n:].decode(errors='replace')

attempts = []
fp = FINGERPRINT
proc = start(fp)
ok, info = wait_healthy(proc, 40 * 60)
attempts.append({'fingerprint': fp, 'prefix_cache': PREFIX_CACHE, 'ok': ok, 'info': info})
if not ok and proc.poll() is not None:
    other = [h for h in re.findall(r'\b[0-9a-f]{64}\b', server_tail()) if h != FINGERPRINT]
    if other:  # Flare named the key it authenticated; accept that one once
        fp = other[-1]
        proc = start(fp)
        ok, info = wait_healthy(proc, 40 * 60)
        attempts.append({'fingerprint': fp, 'prefix_cache': PREFIX_CACHE, 'ok': ok, 'info': info})
if not ok and proc.poll() is not None:  # one recorded fallback: no prefix cache (a lower bound on r)
    proc = start(fp, prefix_cache=None)
    ok, info = wait_healthy(proc, 40 * 60)
    attempts.append({'fingerprint': fp, 'prefix_cache': None, 'ok': ok, 'info': info})
RESULT['start'] = {'attempts': attempts, 'server_tail': server_tail(), 'watchdog': dict((k, v) for k, v in WATCH.items() if k != 'proc')}
smi = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
RESULT['start']['gpu_memory'] = smi
try:
    RESULT['start']['ra_last'] = json.loads((OUT / 'flare_ra.log').read_text().splitlines()[-1])
except Exception as e:
    RESULT['start']['ra_last'] = repr(e)
try:
    RESULT['start']['cu_last'] = json.loads((OUT / 'flare_cu.log').read_text().splitlines()[-1])
except Exception as e:
    RESULT['start']['cu_last'] = repr(e)
log(f'start: {attempts} gpu {smi}'); save()
if not ok:
    log('FLARE did not reach health; stopping (card: INVALID/KILL by cause)')
    raise SystemExit(0)
status, raw = http('GET', '/v1/models'); RESULT['models'] = raw.decode(errors='replace')[:2000]
status, raw = http('GET', '/metrics'); RESULT['metrics_endpoint'] = {'status': status, 'head': raw[:1500].decode(errors='replace')}

# ---- 5. functional: tool calls on logged requests ---------------------------------------------
sys.path.insert(0, str(DATA))
import payload, replay, sim
_HIST_PLACEHOLDER = '[grid image omitted from history]'   # dfranzen tool_agent._HISTORY_IMAGE_PLACEHOLDER
def _is_img(part):
    return isinstance(part, dict) and part.get('type') == 'image_url'
def keep_current_images(messages):
    # 3e: no images at all; every image part becomes the same placeholder, so history never changes
    out = []
    for m in messages:
        c = m.get('content')
        if isinstance(c, list) and any(_is_img(x) for x in c):
            m = {**m, 'content': [{'type': 'text', 'text': _HIST_PLACEHOLDER} if _is_img(x) else x for x in c]}
        out.append(m)
    return out
_full_build = payload.build
def _build_keep0(rec, model, max_tokens):
    body = _full_build(rec, model, max_tokens)
    body['messages'] = keep_current_images(body['messages'])
    return body
payload.build = _build_keep0
replay.payload.build = _build_keep0
_imgs = [sum(_is_img(x) for m in _build_keep0(json.loads(l), MODEL, 1)['messages'] if isinstance(m.get('content'), list) for x in m['content'])
         for path in sorted(DATA.glob('*_p0_requests.jsonl')) for l in open(path) if '"event": "request"' in l]
RESULT['image_window'] = {'keep': 'none', 'requests': len(_imgs), 'max_images': max(_imgs), 'over_4': sum(i > 4 for i in _imgs)}
log(f'image window: {RESULT["image_window"]}'); save()
assert RESULT['image_window']['over_4'] == 0, RESULT['image_window']
records = []
for path in sorted(DATA.glob('*_p0_requests.jsonl'))[:6]:
    with open(path) as f:
        reqs = [json.loads(l) for l in f if '"event": "request"' in l]
    records += [reqs[len(reqs) // 2]]          # mid-game: long context, retained functions
STRIP = []
def build(rec, max_tokens):
    body = payload.build(rec, MODEL, max_tokens)
    for k in STRIP:
        body.pop(k, None)
    return body
def probe_once(rec):
    body = build(rec, 16 * 1024)
    t = time.monotonic()
    status, raw = http('POST', '/v1/chat/completions', body, timeout=900)
    row = {'status': status, 'seconds': round(time.monotonic() - t, 1)}
    try:
        resp = json.loads(raw)
        msg = resp['choices'][0]['message']; calls = msg.get('tool_calls') or []
        args = [json.loads(c['function']['arguments']) for c in calls]
        row.update(finish=resp['choices'][0].get('finish_reason'), usage=resp.get('usage'),
                   tool_calls=len(calls), names=[c['function']['name'] for c in calls],
                   code_ok=bool(args) and all(isinstance(a.get('code'), str) and a['code'].strip() for a in args),
                   reasoning_chars=len(msg.get('reasoning_content') or msg.get('reasoning') or ''),
                   content_head=(msg.get('content') or '')[:300], code_head=(args[0].get('code', '')[:300] if args else ''))
    except Exception as e:
        row['parse_error'] = repr(e)[:300]; row['raw_head'] = raw[:600].decode(errors='replace')
    return row

first = probe_once(records[0])                    # one request alone: also the single-stream speed
if first['status'] == 400:                        # unknown-field rejection: strip vendor extras once
    STRIP[:] = ['top_k', 'chat_template_kwargs']
    first = probe_once(records[0])
functional = [first]
rows = [None] * (len(records) - 1)
def _run(i):
    rows[i] = probe_once(records[i + 1])
ths = [threading.Thread(target=_run, args=(i,)) for i in range(len(rows))]
[th.start() for th in ths]; [th.join() for th in ths]
functional += rows
for row in functional:
    log(f'functional: {row}')
RESULT['functional'] = {'rows': functional, 'stripped_fields': STRIP}; save()
if STRIP:
    _orig = payload.build
    def _stripped(rec, model, max_tokens):
        body = _orig(rec, model, max_tokens)
        for k in STRIP:
            body.pop(k, None)
        return body
    payload.build = _stripped
    replay.payload.build = _stripped

# ---- 6. closed-loop replay at S = 10, 14, 16 ---------------------------------------------------
def u_estimates(events_path):
    ev = [json.loads(l) for l in open(events_path)]
    ph = {e['name']: e['t'] for e in ev if e['kind'] == 'phase'}
    m0, m1 = ph.get('measure_start'), ph.get('measure_end')
    if m0 is None or m1 is None:
        return {'valid': False}
    resp = [e for e in ev if e['kind'] == 'response' and e['status'] == 200]
    inb = [e for e in resp if m0 <= e['t'] <= m1]
    tok = sum(e.get('completion_tokens') or 0 for e in inb)
    ov = sum((e.get('completion_tokens') or 0) * max(0, min(e['t'], m1) - max(e['t_send'], m0)) / (e['t'] - e['t_send'])
             for e in resp if e['t'] > e['t_send'])
    fin = {}
    for e in (x for x in ev if x['kind'] == 'response'):
        k = f"{e['status']}:{e.get('finish_reason')}"; fin[k] = fin.get(k, 0) + 1
    return {'valid': True, 'block_s': m1 - m0, 'responses_block': len(inb), 'U_resp': tok / (m1 - m0),
            'U_overlap': ov / (m1 - m0), 'mean_latency_s': sum(e['t'] - e['t_send'] for e in inb) / max(1, len(inb)),
            'finish': fin, 'errors': sum(1 for e in ev if e['kind'] == 'response' and e['status'] != 200),
            'early_stop': sum(1 for e in resp if e.get('finish_reason') == 'stop'
                              and (e.get('completion_tokens') or 0) < e['recorded_completion'])}

alpha = json.loads((DATA / 'alpha.json').read_text())['alpha']
RESULT['replay'] = {}
for S in (10, 14):
    if DEADLINE - time.monotonic() < 16 * 60:
        log(f'S={S}: skipped, {int((DEADLINE - time.monotonic()) / 60)} min left'); break
    d = OUT / f'S{S}'; d.mkdir(exist_ok=True)
    r = replay.Replay(workload=DATA / 'workload.json', logs_dir=DATA, alpha=alpha,
                      base_url=f'http://127.0.0.1:{PORT}', model=MODEL, events_path=d / 'events.jsonl',
                      slots=S, header={'server': 'flare', 'slots': S, 'stripped_fields': STRIP})
    before = set(threading.enumerate())
    r.run(measure_s=480.0, warmup_max_s=600.0, settle_s=30.0)
    time.sleep(5)
    RESULT['replay'][S] = u_estimates(d / 'events.jsonl')
    try:
        RESULT['replay'][S]['summarize'] = replay.summarize(d / 'events.jsonl')
    except Exception as e:
        RESULT['replay'][S]['summarize'] = repr(e)[:300]
    log(f'S={S}: {RESULT["replay"][S]}'); save()
    # let the stopped replay's in-flight requests finish before the next level starts
    t = time.monotonic()
    while time.monotonic() - t < 900 and [th for th in threading.enumerate() if th not in before]:
        time.sleep(5)
    log(f'S={S}: drained in {time.monotonic() - t:.0f}s')

SGLANG_M3A = {'U_resp': 518.2, 'U_overlap': 477.6}   # 2026-10-08 retry1, same harness, S=10
r10 = RESULT['replay'].get(10, {})
if r10.get('valid'):
    RESULT['ratio_vs_sglang_m3a'] = {k: r10[k] / v for k, v in SGLANG_M3A.items()}
log(f'ratio vs SGLang M3A: {RESULT.get("ratio_vs_sglang_m3a")}')
RESULT['server_tail_end'] = server_tail(6000)
os.killpg(proc.pid, signal.SIGTERM)
save()
